# Lab 23-06 · Programming a Flash Cell and Counting Its Errors

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize, stats
import nand


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · ngspice

*Predict* the steady-state step before running.

In [2]:
sim = ngspice("ispp.cir")
print(f"first pulse: {sim['vt1']:.4f} V")
print(f"steps from pulses 29 and 30: {sim['step29']:.4f} V, {sim['step30']:.4f} V (the pulse step is 0.25 V)")
print(f"total shift after 30 pulses: {sim['vt30']:.4f} V")

first pulse: 1.9174 V
steps from pulses 29 and 30: 0.2501 V, 0.2500 V (the pulse step is 0.25 V)
total shift after 30 pulses: 9.7092 V


## Part 2 · Error rate against level width

*Predict* the σ at which QLC reaches 10⁻³, and how much larger TLC's is.

In [3]:
sig = np.linspace(0.02, 0.2, 200)
plt.figure()
for bits, name in ((1, "SLC"), (2, "MLC"), (3, "TLC"), (4, "QLC")):
    plt.semilogy(sig, [max(nand.rber(6, bits, s), 1e-30) for s in sig], label=name)
plt.ylim(1e-15, 1); plt.xlabel("sigma (V)"); plt.ylabel("raw bit error rate"); plt.legend(); plt.show()
for bits, name in ((3, "TLC"), (4, "QLC")):
    s = optimize.brentq(lambda x: nand.rber(6, bits, x) - 1e-3, 0.01, 0.5)
    print(f"{name}: sigma for a raw error rate of 1e-3 = {s:.4f} V")

TLC: sigma for a raw error rate of 1e-3 = 0.1464 V
QLC: sigma for a raw error rate of 1e-3 = 0.0700 V


## Part 3 · Monte Carlo

*Predict* the error rate of 100,000 QLC cells at σ = 0.08 V from §3.3.

In [4]:
rng = np.random.default_rng(2306)
levels, bits, window, sigma, n = 16, 4, 6.0, 0.08, 100_000
gray = np.array([i ^ (i >> 1) for i in range(levels)])
centres = np.arange(levels) * window / (levels - 1)
sent = rng.integers(0, levels, n)
vt = centres[sent] + rng.normal(0, sigma, n)
refs = (centres[:-1] + centres[1:]) / 2
read = np.searchsorted(refs, vt)
flips = gray[sent] ^ gray[read]
errors = sum(bin(int(f)).count("1") for f in flips)
print(f"Monte Carlo: {errors} bit errors in {n * bits} bits, RBER = {errors / (n * bits):.3e}")
print(f"formula (Example 23-06.4g): {nand.rber(window, bits, sigma):.3e}")

Monte Carlo: 1214 bit errors in 400000 bits, RBER = 3.035e-03
formula (Example 23-06.4g): 2.911e-03
